# World Cup Transit Service near Levi's Stadium

In [ ]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

## Regional Trips

In [ ]:
levi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.bay_area_names,
    route_name_dict = wc_vars.special_bayarea_routes_dict,
    event_time_of_day_dict = wc_vars.levi_match_times  
)

In [ ]:
daily_trips_by_operator = D1.aggregate_daily_trips(
    levi_trips, ["service_date", "schedule_name"]
)

In [ ]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_operator, wc_vars.levi_dates, color_col="schedule_name"
).properties(
    title= "Daily Trips by Operator during World Cup",
    width=500, height=300
)

## Trips by Route

In [ ]:
daily_trips_by_route = D1.aggregate_daily_trips(
    levi_trips, ["service_date", "schedule_name", "route_name"]
)

In [ ]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_route, wc_vars.levi_dates, color_col="route_name"
).properties(
    title= "Daily Trips by Route during World Cup",
    width=500, height=300
)

## Stop Arrivals

In [ ]:
levi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.bay_area_names,
    route_name_dict = wc_vars.special_bayarea_routes_dict,
    event_time_of_day_dict = wc_vars.levi_match_times
)

# Do this separately, because we need stop's pt geom
arrivals_by_event_type = D2.aggregate_by_event_type(levi_stop_arrivals)

In [ ]:
weekday_wide = D2.make_wide(
    arrivals_by_event_type[arrivals_by_event_type.day_type == "weekday"]
).rename(columns = {
    **{c: f"weekday_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
})

weekend_wide = D2.make_wide(
    arrivals_by_event_type[arrivals_by_event_type.day_type == "weekend"]
).rename(columns = {
    **{c: f"weekend_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
})

arrivals_wide = pd.merge(
    weekday_wide,
    weekend_wide,
    on = ["schedule_name", "stop_id", "stop_name"],
    how = "inner"
).pipe(D2.merge_in_stop_geom, levi_stop_arrivals)

In [ ]:
operator_df = (
    arrivals_wide
    .groupby(["schedule_name", "route_id_array", "stop_name"])
    .agg({
        "weekday_change_daily_arrivals": "sum",
        "weekend_change_daily_arrivals": "sum",
        "stop_id": "nunique"
    })
    .reset_index()
    .rename(columns = {"stop_id": "n_stop_ids"})
)

In [ ]:
for i in sorted(operator_df.schedule_name.unique()):
    chart = chart_utils.weekday_weekend_chart_by_operator(operator_df, i)
    display(chart)